# 🧩 Mini-Lab: Read Tracebacks

**Module: Ship a Working AI Feature Today** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Read** a Python traceback bottom-up to find the actual error
2. **Identify** the line in *your own file* that triggered the failure
3. **Distinguish** between library-internal frames and your code frames in a traceback

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Traceback Reading | Read the last line first to see the error type and message, then scan upward to find the line that names your file, that is usually where you should start fixing. |


## Why Tracebacks Matter for AI Engineers

When your AI pipeline breaks, Python gives you a **traceback** — a stack of function calls that led to the error. Most beginners read it top-down and get lost in library code. The professional move is to read it **bottom-up**:

```
Step 1  →  Read the LAST line: what error type + message?
Step 2  →  Scan UP to find the first frame that names YOUR file
Step 3  →  That line number is where you start debugging
```

Let's practice this with real tracebacks — no AI calls needed, just pure Python errors.

In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Anatomy of a Traceback

Here's a typical traceback you might see when building an AI pipeline. Let's label its parts.

In [2]:
SAMPLE_TRACEBACK = '''
Traceback (most recent call last):
  File "/home/user/my_pipeline.py", line 12, in run_pipeline
    result = parse_output(raw_text)
  File "/home/user/my_pipeline.py", line 28, in parse_output
    data = json.loads(raw_text)
  File "/usr/lib/python3.11/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "/usr/lib/python3.11/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "/usr/lib/python3.11/json/decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
'''.strip()

md(
    "### A Real-Looking Traceback\n\n"
    f"```\n{SAMPLE_TRACEBACK}\n```\n\n"
    "**Reading it bottom-up:**\n\n"
    "1. **Last line** → `json.decoder.JSONDecodeError: Expecting value` — the model returned something that isn't valid JSON\n"
    "2. **Scan up past library frames** → the `/usr/lib/python3.11/json/` lines are Python's internal `json` module — not your code\n"
    "3. **Find YOUR file** → `my_pipeline.py`, line 28, in `parse_output` — this is where you call `json.loads()` on bad data\n\n"
    "👉 **Debug line 28 of `my_pipeline.py`**, not anything in `/usr/lib/python3.11/`."
)

### A Real-Looking Traceback

```
Traceback (most recent call last):
  File "/home/user/my_pipeline.py", line 12, in run_pipeline
    result = parse_output(raw_text)
  File "/home/user/my_pipeline.py", line 28, in parse_output
    data = json.loads(raw_text)
  File "/usr/lib/python3.11/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "/usr/lib/python3.11/json/decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "/usr/lib/python3.11/json/decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)
```

**Reading it bottom-up:**

1. **Last line** → `json.decoder.JSONDecodeError: Expecting value` — the model returned something that isn't valid JSON
2. **Scan up past library frames** → the `/usr/lib/python3.11/json/` lines are Python's internal `json` module — not your code
3. **Find YOUR file** → `my_pipeline.py`, line 28, in `parse_output` — this is where you call `json.loads()` on bad data

👉 **Debug line 28 of `my_pipeline.py`**, not anything in `/usr/lib/python3.11/`.

## Trigger Real Tracebacks

Let's cause three common errors and practice reading each traceback bottom-up. We'll write small helper functions that simulate parts of an AI pipeline, so the tracebacks have multiple frames — just like real ones.

### Error 1: `KeyError` — Missing Field

In [3]:
def build_prompt(ticket):
    """Build a prompt from a ticket dict."""
    # Bug: 'subject' doesn't exist in the dict
    return f"Classify this: {ticket['subject']}"

def process_ticket(ticket):
    """Process a single ticket through the pipeline."""
    prompt = build_prompt(ticket)
    return prompt

try:
    process_ticket({"ticket_id": "101", "message": "I was charged twice"})
except KeyError:
    import traceback
    tb_text = traceback.format_exc()
    print(tb_text)

Traceback (most recent call last):
  File "C:\Users\molak\AppData\Local\Temp\ipykernel_37520\1807179058.py", line 12, in <module>
    process_ticket({"ticket_id": "101", "message": "I was charged twice"})
  File "C:\Users\molak\AppData\Local\Temp\ipykernel_37520\1807179058.py", line 8, in process_ticket
    prompt = build_prompt(ticket)
             ^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\molak\AppData\Local\Temp\ipykernel_37520\1807179058.py", line 4, in build_prompt
    return f"Classify this: {ticket['subject']}"
                             ~~~~~~^^^^^^^^^^^
KeyError: 'subject'



In [4]:
md(
    "### Bottom-Up Reading\n\n"
    "1. **Last line** → `KeyError: 'subject'` — we tried to access a key that doesn't exist\n"
    "2. **Your code frame** → `build_prompt`, the line `ticket['subject']`\n"
    "3. **Fix** → change `'subject'` to `'message'` (the key that actually exists)\n\n"
    "📍 This would be a **Stage 3 — Prompt** failure in our six-box pipeline."
)

### Bottom-Up Reading

1. **Last line** → `KeyError: 'subject'` — we tried to access a key that doesn't exist
2. **Your code frame** → `build_prompt`, the line `ticket['subject']`
3. **Fix** → change `'subject'` to `'message'` (the key that actually exists)

📍 This would be a **Stage 3 — Prompt** failure in our six-box pipeline.

### Error 2: `FileNotFoundError` — Bad Path

In [5]:
def load_data(path):
    """Load data from a file."""
    with open(path) as f:
        return f.read()

def ingest(path):
    """Stage 1: Ingest raw data."""
    raw = load_data(path)
    return raw

try:
    ingest("data/nonexistent_file.csv")
except FileNotFoundError:
    import traceback
    tb_text = traceback.format_exc()
    print(tb_text)

Traceback (most recent call last):
  File "C:\Users\molak\AppData\Local\Temp\ipykernel_37520\1761210910.py", line 12, in <module>
    ingest("data/nonexistent_file.csv")
  File "C:\Users\molak\AppData\Local\Temp\ipykernel_37520\1761210910.py", line 8, in ingest
    raw = load_data(path)
          ^^^^^^^^^^^^^^^
  File "C:\Users\molak\AppData\Local\Temp\ipykernel_37520\1761210910.py", line 3, in load_data
    with open(path) as f:
         ^^^^^^^^^^
  File "C:\Users\molak\.virtualenvs\text_llm_agent\Lib\site-packages\IPython\core\interactiveshell.py", line 326, in _modified_open
    return io_open(file, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'data/nonexistent_file.csv'



In [6]:
md(
    "### Bottom-Up Reading\n\n"
    "1. **Last line** → `FileNotFoundError: [Errno 2] No such file or directory: 'data/nonexistent_file.csv'`\n"
    "2. **Your code frame** → `load_data`, the `open(path)` call\n"
    "3. **Fix** → check the file path — does the file exist? Is the spelling correct?\n\n"
    "📍 This would be a **Stage 1 — Ingest** failure."
)

### Bottom-Up Reading

1. **Last line** → `FileNotFoundError: [Errno 2] No such file or directory: 'data/nonexistent_file.csv'`
2. **Your code frame** → `load_data`, the `open(path)` call
3. **Fix** → check the file path — does the file exist? Is the spelling correct?

📍 This would be a **Stage 1 — Ingest** failure.

### Error 3: `TypeError` — Wrong Argument Type

In [7]:
import json

def parse_output(raw_text):
    """Parse JSON from the model's response."""
    return json.loads(raw_text)

def process_response(response_obj):
    """Extract and parse the model response."""
    # Bug: passing the whole dict instead of the text string
    return parse_output(response_obj)

try:
    # Accidentally passing a dict instead of a string
    process_response({"content": '{"category": "Billing"}'})
except TypeError:
    import traceback
    tb_text = traceback.format_exc()
    print(tb_text)

Traceback (most recent call last):
  File "C:\Users\molak\AppData\Local\Temp\ipykernel_37520\1892772312.py", line 14, in <module>
    process_response({"content": '{"category": "Billing"}'})
  File "C:\Users\molak\AppData\Local\Temp\ipykernel_37520\1892772312.py", line 10, in process_response
    return parse_output(response_obj)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\molak\AppData\Local\Temp\ipykernel_37520\1892772312.py", line 5, in parse_output
    return json.loads(raw_text)
           ^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\json\__init__.py", line 339, in loads
    raise TypeError(f'the JSON object must be str, bytes or bytearray, '
TypeError: the JSON object must be str, bytes or bytearray, not dict



In [8]:
md(
    "### Bottom-Up Reading\n\n"
    "1. **Last line** → `TypeError: the JSON object must be str, bytes or bytearray, not dict`\n"
    "2. **Your code frame** → `parse_output`, the `json.loads(raw_text)` call — but the real bug is one frame higher\n"
    "3. **The actual mistake** → `process_response` passes the whole dict instead of `response_obj['content']`\n\n"
    "📍 This would be a **Stage 5 — Parse Output** failure.\n\n"
    "> **Key insight:** sometimes the frame that *throws* the error isn't the frame that *caused* it. "
    "Scan up one more frame to find where the wrong value was passed."
)

### Bottom-Up Reading

1. **Last line** → `TypeError: the JSON object must be str, bytes or bytearray, not dict`
2. **Your code frame** → `parse_output`, the `json.loads(raw_text)` call — but the real bug is one frame higher
3. **The actual mistake** → `process_response` passes the whole dict instead of `response_obj['content']`

📍 This would be a **Stage 5 — Parse Output** failure.

> **Key insight:** sometimes the frame that *throws* the error isn't the frame that *caused* it. Scan up one more frame to find where the wrong value was passed.

## The 3-Step Rule

Let's codify the bottom-up reading technique into a reusable checklist.

In [9]:
md(
    "### 🔍 The 3-Step Traceback Rule\n\n"
    "| Step | Action | What to Look For |\n"
    "|------|--------|---------------------|\n"
    "| 1 | Read the **last line** | Error type + message (e.g., `KeyError: 'subject'`) |\n"
    "| 2 | Scan **up** past library frames | Skip lines with paths like `/usr/lib/`, `site-packages/`, or `lib/python3.x/` |\n"
    "| 3 | Find the first frame with **your filename** | That file + line number is where you start debugging |\n\n"
    "> **Library frames** contain paths like `site-packages/openai/`, `/usr/lib/python3.11/json/`, etc.\n"
    "> **Your frames** contain paths like `my_pipeline.py`, `app.py`, or a notebook cell reference like `<ipython-input-5>`.\n\n"
    "In a Jupyter notebook, your code frames show up as `<ipython-input-N>` or `[N]` — those are *your* frames."
)

### 🔍 The 3-Step Traceback Rule

| Step | Action | What to Look For |
|------|--------|---------------------|
| 1 | Read the **last line** | Error type + message (e.g., `KeyError: 'subject'`) |
| 2 | Scan **up** past library frames | Skip lines with paths like `/usr/lib/`, `site-packages/`, or `lib/python3.x/` |
| 3 | Find the first frame with **your filename** | That file + line number is where you start debugging |

> **Library frames** contain paths like `site-packages/openai/`, `/usr/lib/python3.11/json/`, etc.
> **Your frames** contain paths like `my_pipeline.py`, `app.py`, or a notebook cell reference like `<ipython-input-5>`.

In a Jupyter notebook, your code frames show up as `<ipython-input-N>` or `[N]` — those are *your* frames.

## Practice: Parse a Traceback Programmatically

Let's write a tiny helper that extracts the error type and your code's frame from a traceback string. This reinforces the bottom-up reading skill.

In [10]:
def read_traceback_bottomup(tb_string):
    """Extract the key info from a traceback using the 3-step rule."""
    lines = tb_string.strip().splitlines()

    # Step 1: Last line = error type and message
    error_line = lines[-1]

    # Step 2 & 3: Scan upward for the first "File" line that ISN'T a library
    library_markers = ["site-packages", "/usr/lib/", "lib/python", "\\lib\\"]
    your_frame = None

    for line in lines:
        if line.strip().startswith('File'):
            is_library = any(marker in line for marker in library_markers)
            if not is_library:
                your_frame = line.strip()

    return {
        "error": error_line,
        "your_frame": your_frame or "(no user-code frame found)"
    }

# Test it on the sample traceback from earlier
result = read_traceback_bottomup(SAMPLE_TRACEBACK)

md(
    "### Parsed Result\n\n"
    f"**Step 1 — Error:** `{result['error']}`\n\n"
    f"**Step 3 — Your code:** `{result['your_frame']}`\n\n"
    "👉 Start debugging at that file and line number."
)

### Parsed Result

**Step 1 — Error:** `json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)`

**Step 3 — Your code:** `File "/home/user/my_pipeline.py", line 28, in parse_output`

👉 Start debugging at that file and line number.

## 🎯 Summary

### Key Takeaways

1. **Read bottom-up** — the last line of a traceback tells you *what* went wrong (error type + message); start there, not at the top
2. **Skip library frames** — lines with `site-packages/`, `/usr/lib/`, or `lib/python` are internal code you didn't write; scan past them
3. **Find your file** — the first frame that names *your* file (or `<ipython-input>` in a notebook) is where you start debugging